# 31.01 Локальная чувствительность двуслойной модели

> **Статус:** канонический синтетический расчёт локальных производных и
> эластичностей. Экспериментальные данные не используются; приведённые числа не
> являются оценками тканей добровольцев.

Вход — проверенное ядро `two_layer_model.py`/`30.04`. Теоретические определения
даны в `31.00`.

## 1. Определения и условность

Для параметра $p$:

$$S_p=\frac{p}{Z}\frac{\partial Z}{\partial p}.$$

Из однородности по сопротивлениям следует
$S_{\rho_1}+S_{\rho_2}=1$. Если только $\rho_2$ оценивается из одного
наблюдения, а ошибка $h$ ошибочно не учитывается, условный перенос равен

$$\left.\frac{\partial\ln\rho_2}{\partial\ln h}\right|_Z
=-\frac{S_h}{S_{\rho_2}}.$$

Обозначим этот знаковый локальный коэффициент переноса как
$K_h=-S_h/S_{\rho_2}$. Для оценки величины неопределённости используют
$|K_h|$.

Это не общая погрешность обратной задачи. Формула предполагает фиксированные
$\rho_1$, геометрию, коэффициент усиления, смещение и корректную модель.
При нескольких неизвестных нужен полный якобиан `31.03`.

In [1]:
from pathlib import Path
import sys

import numpy as np

candidates = [Path.cwd(), Path.cwd() / "Colab Notebooks", Path.cwd().parent]
notebook_root = next((p for p in candidates if (p / "two_layer_model.py").exists()), None)
if notebook_root is None:
    raise FileNotFoundError("two_layer_model.py not found; run from the repository or Colab Notebooks directory")
sys.path.insert(0, str(notebook_root))

from two_layer_model import evaluate, geometry_from_size

## 2. Синтетическая рабочая точка

Параметры ниже выбраны только для воспроизводимой проверки вычислений. Любая
экспериментальная интерпретация требует принятой рабочей точки `30.02`,
калибровки и контроля качества. В коде $\rho_1,\rho_2$ заданы в Ом·м,
$h$ и $L$ — в метрах, а $\beta$ безразмерна. Столбец `n_terms` показывает
число членов ряда, потребовавшихся критерию сходимости ядра.

In [2]:
rho1, rho2, h, beta = 5.0, 20.0, 0.020, 0.5
sizes = np.arange(0.050, 0.141, 0.010)

print("L_mm S_rho1 S_rho2 S_h K_h n_terms")
for size in sizes:
    a, b = geometry_from_size(size, beta)
    result = evaluate(rho1, rho2, h, a, b)
    s_rho1 = rho1 / result.z * result.d_rho1
    s_rho2 = rho2 / result.z * result.d_rho2
    s_h = h / result.z * result.d_h
    k_h = -s_h / s_rho2
    assert np.isclose(s_rho1 + s_rho2, 1.0, atol=1e-10)
    print(f"{size*1000:4.0f} {s_rho1:7.4f} {s_rho2:7.4f} {s_h:7.4f} {k_h:7.3f} {result.n_terms:4d}")

L_mm S_rho1 S_rho2 S_h K_h n_terms
  50  0.9220  0.0780 -0.2788   3.575  128
  60  0.8898  0.1102 -0.3565   3.236  128
  70  0.8573  0.1427 -0.4190   2.937  128
  80  0.8258  0.1742 -0.4661   2.676  128
  90  0.7959  0.2041 -0.4996   2.448  128
 100  0.7678  0.2322 -0.5220   2.248  128
 110  0.7417  0.2583 -0.5355   2.073  128
 120  0.7173  0.2827 -0.5424   1.919  128
 130  0.6945  0.3055 -0.5443   1.782  128
 140  0.6733  0.3267 -0.5426   1.661  128


## 3. Геометрическая чувствительность

$S_a$ и $S_b$ описывают раздельные ошибки полубаз. При ручной наклейке ошибка
обычно затрагивает несколько координат и общий центр одновременно, поэтому
эти две производные не являются готовой моделью ошибки установки. Нужен
Якобиан по фактическим координатам либо сценарии КТ/FEM. Тождество
$S_a+S_b+S_h=-1$ ниже проверяет одновременное масштабирование идеальной
геометрии; оно не означает, что реальные ошибки установки складываются таким
образом.

In [3]:
size = 0.140
a, b = geometry_from_size(size, beta)
result = evaluate(rho1, rho2, h, a, b)
s_a = a / result.z * result.d_a
s_b = b / result.z * result.d_b
s_h = h / result.z * result.d_h
assert np.isclose(s_a + s_b + s_h, -1.0, atol=1e-10)
print({"S_a": s_a, "S_b": s_b, "S_h": s_h})

{'S_a': -1.8384253184263442, 'S_b': 1.380995419086554, 'S_h': -0.5425701006602105}


## 4. Выход и границы

Выход: вычислимые $Z$, первые производные и эластичности для заданной рабочей
точки. Они являются локальными свойствами идеальной модели.

Не являются выходом: распределение погрешности, условная граница
Крамера—Рао, оптимальная пара, абсолютные свойства тканей или локализация
источника пульсовой волны. Эти задачи требуют `31.02–31.04`, `32`, данных,
прошедших контроль качества, и выбранного оператора наблюдения (знак или
модуль).